# 01: 从零构建Naive RAG系统

## 本Notebook将带你做什么

我们将从零开始，逐步构建一个**最小可行RAG（Naive RAG）系统**，完整实现学术界定义的RAG四阶段流水线：

```
  [文档] --> 1.索引(Index) --> 2.检索(Retrieve) --> 3.增强(Augment) --> 4.生成(Generate) --> [答案]
```

### 学习目标

1. **深刻理解RAG流水线的每个阶段**：知道每个阶段输入什么、输出什么、做了什么
2. **亲手实现一个完整可运行的Naive RAG**：无需任何RAG框架，纯Python + OpenAI API
3. **理解Naive RAG的局限性**：通过真实失败案例，体会为什么我们需要Phase 04的深度优化
4. **为后续Phase打下坚实基础**：这些概念是理解LangChain、LlamaIndex、生产级RAG的前提

### 你需要准备

- 一个OpenAI API Key（设置在环境变量 `OPENAI_API_KEY` 中）
- Python 3.8+
- 基础的Python和机器学习概念（向量、余弦相似度）

让我们开始吧！

## 1. 环境准备

在这一步，我们导入所有需要的库。如果你还没有安装这些依赖，请先运行以下命令：

```bash
pip install openai numpy
```

我们只需要两个核心库：
- **openai**：调用OpenAI的Embedding和Chat Completion API
- **numpy**：进行向量运算（余弦相似度）


In [ ]:
# pip install openai numpy  (uncomment and run if not installed)

import os
import json
import time
import numpy as np
from typing import List, Dict, Optional, Tuple
from openai import OpenAI

print("All imports successful!")
print(f"NumPy version: {np.__version__}")

## 2. MinimalRAG类 - 索引阶段

### 什么是指数（Index）阶段？

索引阶段是RAG流水线的**第一步**，也是最容易被忽略的一步。它的核心任务只有一个：

> **将文档集合转化为可供语义搜索的向量索引**

具体来说，索引阶段包含：
1. **文档分块**（Naive RAG中，每个文档就是一个块）
2. **向量化（Embedding）**：将每段文本映射为一个高维向量
3. **存储**：保存「原文 + 向量」对，供后续检索使用

### 在Naive RAG中

我们简化处理：每个文档作为一个整体进行Embedding。这意味着**没有分块策略**、**没有元数据管理**、**没有增量更新**——这就是"Naive"的来源之一。

下面我们实现 `MinimalRAG` 的 `__init__` 和 `index()` 方法。

In [ ]:
class MinimalRAG:
    """
    A minimal, naive implementation of Retrieval-Augmented Generation (RAG).
    
    This implementation follows the 4-stage RAG pipeline:
        1. Index  - Embed documents into a vector store
        2. Retrieve - Find the most relevant documents via cosine similarity
        3. Augment - Combine the query with retrieved context into a prompt
        4. Generate - Call the LLM to produce a grounded answer
    
    The "naive" label reflects simplifications:
        - No document chunking (each doc = one chunk)
        - No metadata filtering
        - No reranking
        - No query rewriting
        - In-memory storage only
    """
    
    def __init__(self, openai_api_key: str, embedding_model: str = "text-embedding-3-small"):
        """
        Initialize the MinimalRAG system.
        
        Args:
            openai_api_key: Your OpenAI API key.
            embedding_model: The OpenAI embedding model to use.
        """
        self.client = OpenAI(api_key=openai_api_key)
        self.embedding_model = embedding_model
        self.documents: List[Dict] = []  # list of {"content": str, "embedding": np.ndarray}
        self._indexed = False
        print(f"MinimalRAG initialized with embedding model: {embedding_model}")
    
    def index(self, documents: List[str]) -> None:
        """
        Index a list of documents by computing and storing their embeddings.
        
        Each document is treated as a single chunk (no splitting).
        Embeddings are obtained via the OpenAI Embeddings API.
        
        Args:
            documents: A list of document strings to index.
            
        Raises:
            RuntimeError: If the OpenAI API call fails after retries.
        """
        if not documents:
            print("Warning: Empty document list provided. Nothing to index.")
            return
        
        print(f"Indexing {len(documents)} documents...")
        
        for i, doc in enumerate(documents):
            attempt = 0
            max_retries = 3
            success = False
            
            while attempt < max_retries and not success:
                try:
                    response = self.client.embeddings.create(
                        model=self.embedding_model,
                        input=doc
                    )
                    embedding = np.array(response.data[0].embedding, dtype=np.float32)
                    self.documents.append({
                        "content": doc,
                        "embedding": embedding
                    })
                    success = True
                    print(f"  [{i+1}/{len(documents)}] Indexed: '{doc[:50]}...' (dim={len(embedding)})")
                    
                except Exception as e:
                    attempt += 1
                    wait_time = 2 ** attempt  # Exponential backoff: 2, 4, 8 seconds
                    print(f"  Error indexing doc {i+1} (attempt {attempt}/{max_retries}): {e}")
                    if attempt < max_retries:
                        print(f"  Retrying in {wait_time}s...")
                        time.sleep(wait_time)
                    else:
                        raise RuntimeError(
                            f"Failed to index document after {max_retries} attempts: {e}"
                        )
        
        self._indexed = True
        print(f"Indexing complete. {len(self.documents)} documents stored.")
    
    @property
    def is_indexed(self) -> bool:
        """Check whether documents have been indexed."""
        return self._indexed and len(self.documents) > 0

print("MinimalRAG class with __init__ and index() defined successfully!")

## 3. MinimalRAG类 - 检索阶段

### 什么是检索（Retrieve）阶段？

检索阶段是RAG流水线的**核心差异化能力**。当我们收到用户查询时，我们需要：

1. 将**查询向量化**（与文档使用同一个Embedding模型）
2. 计算查询向量与所有文档向量之间的**相似度**
3. 返回**最相似的Top-K文档**

### 余弦相似度（Cosine Similarity）

我们使用余弦相似度来衡量两个向量的相似程度：

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

- 值域：[-1, 1]，越接近1表示越相似
- 与向量长度无关，只关心方向——这对文本Embedding很重要

### Naive RAG的检索

我们使用最基本的检索策略：
- **暴力搜索**：遍历所有文档，逐一计算余弦相似度
- **无重排序**：直接按原始相似度排序返回
- **无查询改写**：原样使用用户的查询进行检索

（这就是"Naive"的第二个来源——在生产环境中，你会用向量数据库+重排序+查询优化）

In [ ]:
# Extend MinimalRAG with the retrieve() method

def _retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
    """
    Retrieve the top-k most relevant documents for a given query.
    
    Uses cosine similarity between the query embedding and all document
    embeddings to find the most semantically similar documents.
    
    Args:
        query: The user's query string.
        top_k: The number of top documents to return.
    
    Returns:
        A list of dicts, each with 'content' (str) and 'score' (float),
        sorted by descending similarity score.
    
    Raises:
        ValueError: If no documents have been indexed yet.
        RuntimeError: If the query embedding API call fails.
    """
    if not self.is_indexed:
        raise ValueError(
            "No documents indexed. Call index() before retrieve()."
        )
    
    # Step 1: Embed the query
    print(f"Embedding query: '{query}'")
    try:
        response = self.client.embeddings.create(
            model=self.embedding_model,
            input=query
        )
        query_embedding = np.array(response.data[0].embedding, dtype=np.float32)
    except Exception as e:
        raise RuntimeError(f"Failed to embed query: {e}")
    
    # Step 2: Compute cosine similarity with every document
    # cosine(A, B) = dot(A, B) / (||A|| * ||B||)
    similarities = []
    query_norm = np.linalg.norm(query_embedding)
    
    for i, doc in enumerate(self.documents):
        doc_embedding = doc["embedding"]
        doc_norm = np.linalg.norm(doc_embedding)
        
        if query_norm == 0 or doc_norm == 0:
            score = 0.0
        else:
            dot_product = np.dot(query_embedding, doc_embedding)
            score = dot_product / (query_norm * doc_norm)
        
        similarities.append({
            "content": doc["content"],
            "score": float(score)
        })
    
    # Step 3: Sort by similarity score (descending) and take top_k
    similarities.sort(key=lambda x: x["score"], reverse=True)
    top_results = similarities[:top_k]
    
    print(f"Retrieved top {len(top_results)} documents:")
    for i, result in enumerate(top_results):
        print(f"  [{i+1}] score={result['score']:.4f} | '{result['content'][:60]}...'")
    
    return top_results


# Monkey-patch the method onto the class
MinimalRAG.retrieve = _retrieve

print("retrieve() method added to MinimalRAG successfully!")

## 4. MinimalRAG类 - 生成阶段

### 什么是生成（Generate）阶段？

生成阶段是RAG流水线的**最终输出阶段**。检索到的相关文档被整合进一个提示词（Prompt），然后LLM基于这个「增强后的提示词」生成答案。

这一阶段包含两个子步骤：

1. **增强（Augment）**：将检索到的文档和用户查询拼装成一个结构化的Prompt
2. **生成（Generate）**：将Prompt发送给LLM，获取最终答案

### 提示词设计

一个好的RAG Prompt需要告诉LLM：
- **角色**：你是一个什么样的助手
- **任务**：基于给定资料回答问题
- **约束**：只能使用资料中的信息，不知道就说不知道——这是**减少幻觉的关键**
- **格式**：如何呈现检索到的上下文

In [ ]:
# Extend MinimalRAG with the generate() method

def _generate(
    self,
    query: str,
    retrieved_docs: List[Dict],
    model: str = "gpt-3.5-turbo",
    temperature: float = 0.0
) -> str:
    """
    Generate an answer based on the query and retrieved context documents.
    
    Constructs a Chinese-language prompt that instructs the LLM to:
        - Answer only using the provided context
        - Say "I don't know" if the context is insufficient
        - Cite which sources were used
    
    Args:
        query: The user's question.
        retrieved_docs: List of retrieved documents from retrieve().
        model: The OpenAI chat model to use for generation.
        temperature: Sampling temperature (0.0 = deterministic).
    
    Returns:
        The generated answer string.
    
    Raises:
        RuntimeError: If the OpenAI API call fails after retries.
    """
    # Step 1: Build the context block from retrieved documents
    if not retrieved_docs:
        context_block = "（无相关文档被检索到）"
    else:
        context_parts = []
        for i, doc in enumerate(retrieved_docs):
            context_parts.append(
                f"[文档{i+1}] (相关度: {doc['score']:.3f})\n{doc['content']}"
            )
        context_block = "\n\n".join(context_parts)
    
    # Step 2: Construct the system prompt (Chinese)
    system_prompt = (
        "你是一个严谨的信息检索助手。你的任务是根据提供的参考资料回答用户问题。\n\n"
        "请严格遵守以下规则：\n"
        "1. 仅使用【参考资料】中明确包含的信息来回答问题。\n"
        "2. 如果【参考资料】中没有足够信息来回答问题，请明确说\"根据提供的资料，我无法回答这个问题\"。\n"
        "3. 不要编造任何信息。宁可说不知道，也不要猜测。\n"
        "4. 如果使用了多个文档的信息，请综合回答。\n"
        "5. 回答用中文，简洁明了。"
    )
    
    # Step 3: Construct the user message with context and query
    user_message = (
        f"【参考资料】\n"
        f"{context_block}\n\n"
        f"【用户问题】\n"
        f"{query}\n\n"
        f"请基于以上参考资料回答问题。"
    )
    
    # Step 4: Call the OpenAI Chat Completions API
    attempt = 0
    max_retries = 3
    
    while attempt < max_retries:
        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=temperature,
                max_tokens=500
            )
            answer = response.choices[0].message.content
            if answer is None:
                answer = "（API 返回空内容）"
            
            print(f"Generation complete. Token usage: {response.usage}")
            return answer.strip()
            
        except Exception as e:
            attempt += 1
            wait_time = 2 ** attempt
            print(f"  Generation error (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(
                    f"Failed to generate answer after {max_retries} attempts: {e}"
                )


# Monkey-patch the method onto the class
MinimalRAG.generate = _generate

print("generate() method added to MinimalRAG successfully!")

## 5. MinimalRAG类 - 完整流水线

### 将所有阶段串联起来

现在我们来实现 `query()` 方法——它将索引之后的三个阶段串联成一个完整的流水线：

```
query(user_question)
  --> retrieve(user_question, top_k)     # 检索相关文档
  --> generate(user_question, docs)      # 基于文档生成答案
  --> return answer                       # 返回最终结果
```

一个设计良好的 `query()` 方法应该：
1. 有清晰的错误处理（API失败、空文档等）
2. 有适度的日志输出，让用户知道进度
3. 返回结构化的结果（不仅仅是答案，还有检索到的文档）


In [ ]:
def _query(
    self,
    query: str,
    top_k: int = 3,
    model: str = "gpt-3.5-turbo",
    temperature: float = 0.0
) -> Dict:
    """
    Run the complete RAG pipeline: retrieve + generate.
    
    This is the main entry point for end users. It chains together:
        1. retrieve() - Find relevant documents
        2. generate() - Generate an answer grounded in those documents
    
    Args:
        query: The user's question.
        top_k: Number of documents to retrieve.
        model: OpenAI chat model for generation.
        temperature: Sampling temperature.
    
    Returns:
        A dict with keys:
            - 'query' (str): The original query
            - 'retrieved_docs' (list): Retrieved documents with scores
            - 'answer' (str): The generated answer
            - 'pipeline_time' (float): Total time in seconds
    """
    print(f"\n{'='*60}")
    print(f"RAG Pipeline - Query: '{query}'")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    # Stage 1: Retrieve
    print("\n[Stage 2/3] Retrieving relevant documents...")
    try:
        retrieved_docs = self.retrieve(query=query, top_k=top_k)
    except Exception as e:
        print(f"ERROR in retrieval: {e}")
        return {
            "query": query,
            "retrieved_docs": [],
            "answer": f"检索失败: {e}",
            "pipeline_time": time.time() - start_time
        }
    
    # Stage 2: Generate
    print(f"\n[Stage 3/3] Generating answer with {model}...")
    try:
        answer = self.generate(
            query=query,
            retrieved_docs=retrieved_docs,
            model=model,
            temperature=temperature
        )
    except Exception as e:
        print(f"ERROR in generation: {e}")
        answer = f"生成失败: {e}"
    
    pipeline_time = time.time() - start_time
    
    print(f"\nPipeline completed in {pipeline_time:.2f}s")
    print(f"{'='*60}\n")
    
    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "answer": answer,
        "pipeline_time": pipeline_time
    }


# Monkey-patch the method onto the class
MinimalRAG.query = _query

print("query() method added to MinimalRAG successfully!")
print("MinimalRAG class is now COMPLETE with all 4 pipeline stages.")

## 6. 测试：在小文档集上运行

现在我们来测试我们构建的系统。我们将：

1. 创建一个包含5-8篇中文文档的小型文档集
2. 初始化 MinimalRAG
3. 索引文档
4. 运行3个测试查询
5. 查看每个查询的检索结果和生成答案

### 文档集设计

我们的文档集涵盖多个AI相关主题（但每个主题独立成篇），这样我们可以测试：
- 精确匹配查询（文档明确包含答案）
- 跨文档查询（需要综合多篇文档的信息）
- 边界查询（文档中不包含的信息，期待RAG说"不知道"）

**注意**：请确保你已经设置了 `OPENAI_API_KEY` 环境变量。如果还没有，请在终端运行：
```bash
export OPENAI_API_KEY="sk-your-key-here"
```
或者在代码中直接传入（不推荐，安全性差）。

In [ ]:
# ============================================================
# Step 1: Create a test document set
# ============================================================

test_documents = [
    # Document 1: Machine Learning
    "机器学习是人工智能的一个分支，它使计算机系统能够从数据中自动学习和改进，"
    "而无需显式编程。常见的机器学习方法包括监督学习、无监督学习和强化学习。"
    "监督学习使用带有标签的数据进行训练，典型算法包括线性回归、决策树和支持向量机（SVM）。",
    
    # Document 2: Deep Learning
    "深度学习是机器学习的一个子领域，它使用多层人工神经网络来学习数据的层次化表示。"
    "深度学习在图像识别、语音识别和自然语言处理等领域取得了突破性成果。"
    "最流行的深度学习框架包括PyTorch和TensorFlow。卷积神经网络（CNN）特别擅长处理图像数据，"
    "而循环神经网络（RNN）和Transformer架构擅长处理序列数据。",
    
    # Document 3: Natural Language Processing
    "自然语言处理（NLP）是人工智能的一个重要领域，致力于让计算机理解、生成和处理人类语言。"
    "NLP的核心任务包括文本分类、命名实体识别（NER）、情感分析、机器翻译和问答系统。"
    "近年来，基于Transformer架构的大规模预训练语言模型（如BERT、GPT系列）极大地推动了NLP的发展。"
    "词嵌入（Word Embedding）和上下文嵌入是NLP中的关键技术，它们将文本转化为向量表示。",
    
    # Document 4: Computer Vision
    "计算机视觉是人工智能的一个领域，使计算机能够从图像和视频中获取高层次的语义理解。"
    "主要任务包括图像分类、目标检测（Object Detection）、图像分割（Image Segmentation）和图像生成。"
    "YOLO（You Only Look Once）是一种流行的实时目标检测算法。"
    "近年来，扩散模型（Diffusion Models）如Stable Diffusion在图像生成领域取得了惊人成果。"
    "计算机视觉广泛应用于自动驾驶、医疗影像分析和安防监控等领域。",
    
    # Document 5: Reinforcement Learning
    "强化学习是一种机器学习范式，智能体（Agent）通过与环境交互来学习最优行为策略。"
    "智能体根据当前状态选择动作，环境返回奖励（Reward）和下一个状态。"
    "强化学习的目标是最大化累积奖励。深度强化学习结合了深度学习和强化学习，"
    "代表性算法包括DQN（Deep Q-Network）、PPO（Proximal Policy Optimization）和A3C。"
    "AlphaGo和AlphaZero是强化学习的标志性应用，分别在围棋和国际象棋上击败了人类世界冠军。",
    
    # Document 6: Large Language Models
    "大语言模型（LLM）是基于Transformer架构的大规模预训练语言模型，"
    "通常包含数百亿甚至数千亿个参数。代表性的LLM包括OpenAI的GPT-4、Anthropic的Claude、"
    "Google的Gemini和Meta的Llama系列。LLM通过在海量文本数据上进行预训练，"
    "获得了广泛的世界知识和强大的语言理解与生成能力。然而，LLM也存在幻觉（Hallucination）问题，"
    "即生成的内容看似合理但实际上是错误的，这正是RAG技术要解决的核心问题之一。",
    
    # Document 7: RAG (Retrieval-Augmented Generation)
    "检索增强生成（RAG）是一种将信息检索与语言模型生成相结合的技术框架。"
    "RAG的基本流程包括：首先将知识库文档转化为向量索引（Indexing），"
    "然后根据用户查询检索最相关的文档片段（Retrieval），"
    "最后将检索结果作为上下文提供给语言模型进行生成（Generation）。"
    "RAG可以有效减少大语言模型的幻觉问题，使生成的回答更加准确和可靠。"
    "此外，RAG还支持知识的动态更新，无需重新训练模型即可使用最新信息。"
]

print(f"Created test document set with {len(test_documents)} documents.")
for i, doc in enumerate(test_documents):
    print(f"  Doc {i+1}: {doc[:60]}...")


# ============================================================
# Step 2: Initialize MinimalRAG with the API key
# ============================================================

# Try to get the API key from environment variable
api_key = os.environ.get("OPENAI_API_KEY", None)

if api_key is None:
    print("\n" + "="*60)
    print("WARNING: OPENAI_API_KEY environment variable not set!")
    print("="*60)
    print("To run this notebook, you need an OpenAI API key.")
    print("\nOption 1 (recommended): Set the environment variable:")
    print("  export OPENAI_API_KEY='sk-your-key-here'")
    print("  Then restart the notebook kernel.")
    print("\nOption 2 (not recommended for security):")
    print("  Replace the line below with: api_key = 'sk-your-key-here'")
    print("="*60 + "\n")
    
    # Provide a placeholder - the user must fill this in
    # api_key = "sk-your-key-here"  # <-- UNCOMMENT AND REPLACE WITH YOUR KEY
else:
    print(f"API key found (starts with: {api_key[:7]}...)")


# Only proceed if we have a key
if api_key:
    rag = MinimalRAG(openai_api_key=api_key)
else:
    print("\nSkipping RAG initialization - no API key available.")
    print("The code below is ready to run once you set up your key.")
    rag = None


# ============================================================
# Step 3: Index the documents
# ============================================================

if rag is not None:
    rag.index(test_documents)
    print(f"\nIndex status: {rag.is_indexed}")
else:
    print("\n(Skipping indexing - no API key)")


# ============================================================
# Step 4: Run test queries
# ============================================================

test_queries = [
    # Query 1: Direct match - answer is clearly in one document
    "什么是强化学习？它的目标是什么？",
    
    # Query 2: Cross-document - requires info from multiple docs
    "机器学习和深度学习有什么关系？",
    
    # Query 3: Explicit knowledge that should be covered
    "大语言模型存在什么主要问题？RAG如何帮助解决？"
]

if rag is not None:
    results = []
    for i, q in enumerate(test_queries):
        print(f"\n{'#'*60}")
        print(f"# TEST QUERY {i+1}/{len(test_queries)}")
        print(f"{'#'*60}")
        
        result = rag.query(query=q, top_k=3)
        results.append(result)
        
        print(f"\nANSWER: {result['answer']}")
        print(f"Time: {result['pipeline_time']:.2f}s")
        print("-" * 60)
else:
    print("\n(Skipping test queries - no API key)")

## 7. 演示Naive RAG的失败模式

### 为什么"Naive" RAG还不够好？

"Naive"意味着我们做了大量简化：没有文档分块、没有重排序、没有查询改写、没有HyDE、没有元数据过滤……

这些简化在实际应用中会导致严重问题。下面我们演示三种最常见的失败模式：

1. **幻觉（Hallucination）**：查询文档中不存在的知识，LLM仍然自信编造
2. **缺失上下文（Missing Context）**：跨文档信息分散在不同chunk中，检索只返回部分
3. **不相关检索（Irrelevant Retrieval）**：由于Embedding局限，检索到语义不匹配的文档

理解这些失败模式，是理解Phase 04优化路径的前提。

In [ ]:
if rag is not None:
    print("=" * 70)
    print("NAIVE RAG FAILURE MODE DEMONSTRATIONS")
    print("=" * 70)
    
    # ================================================================
    # FAILURE MODE 1: Hallucination
    # ================================================================
    # Our documents contain NO information about "quantum computing".
    # A good RAG system should say "I don't know". A naive one might
    # still generate a confident but fabricated answer.
    # ================================================================
    
    print("\n" + "=" * 70)
    print("FAILURE MODE 1: Hallucination (No Relevant Docs)")
    print("=" * 70)
    print("Query: '量子计算在机器学习中有哪些应用？'")
    print("Expected behavior: Say 'I don't know' (no docs about quantum computing)")
    print("-" * 70)
    
    query_hallucination = "量子计算在机器学习中有哪些应用？"
    
    # First, show what was retrieved
    retrieved_h = rag.retrieve(query_hallucination, top_k=3)
    print("\nRetrieved documents (should be irrelevant to quantum computing):")
    for i, doc in enumerate(retrieved_h):
        print(f"  [{i+1}] score={doc['score']:.4f}: {doc['content'][:80]}...")
    
    # Now generate
    answer_h = rag.generate(query_hallucination, retrieved_h)
    print(f"\nGENERATED ANSWER: {answer_h}")
    print("\nANALYSIS:")
    print("  - The retrieved docs are about ML/DL, not quantum computing.")
    print("  - If the LLM answers with specific quantum computing applications,")
    print("    it is HALLUCINATING - those facts are not in our documents.")
    print("  - A proper RAG prompt can mitigate this, but naive retrieval")
    print("    may still return high-similarity wrong docs that confuse the LLM.")
    
    
    # ================================================================
    # FAILURE MODE 2: Missing Context (Information Scattered)
    # ================================================================
    # When related information is spread across multiple documents
    # (e.g., ML basics in doc 1, DL details in doc 2), retrieving only
    # top-K may miss some relevant docs if K is too small or if the
    # embedding similarity is biased toward one aspect.
    # ================================================================
    
    print("\n\n" + "=" * 70)
    print("FAILURE MODE 2: Missing Context (Information Scattered)")
    print("=" * 70)
    print("Query: '深度学习使用了哪些神经网络架构？各有什么特点？'")
    print("Expected: Need CNN info (Doc 2) + RNN/Transformer info (Doc 2).")
    print("But also NLP doc 3 and CV doc 4 have related context.")
    print("-" * 70)
    
    query_scattered = "深度学习使用了哪些神经网络架构？各有什么特点？"
    
    # Retrieve with different top_k to show the effect
    for k in [1, 2, 5]:
        print(f"\n--- top_k = {k} ---")
        retrieved_s = rag.retrieve(query_scattered, top_k=k)
        for i, doc in enumerate(retrieved_s):
            print(f"  [{i+1}] score={doc['score']:.4f}: {doc['content'][:80]}...")
        
        answer_s = rag.generate(query_scattered, retrieved_s)
        print(f"  Answer (top_k={k}): {answer_s[:150]}...")
    
    print("\nANALYSIS:")
    print("  - With top_k=1, the LLM only sees ONE document's context.")
    print("  - Information about different architectures is spread across docs.")
    print("  - A small top_k means the answer is INCOMPLETE.")
    print("  - But a large top_k introduces noise and increases cost/latency.")
    print("  - This is the fundamental retrieval-quality tradeoff.")
    
    
    # ================================================================
    # FAILURE MODE 3: Irrelevant Retrieval
    # ================================================================
    # Embedding models can be fooled by superficial keyword overlap
    # rather than true semantic similarity. We craft a query that
    # uses words that appear in multiple documents but in a different
    # context than intended.
    # ================================================================
    
    print("\n\n" + "=" * 70)
    print("FAILURE MODE 3: Irrelevant Retrieval (Embedding Limitations)")
    print("=" * 70)
    print("Query: 'AlphaGo使用的深度学习技术有哪些？'")
    print("Expected: Should retrieve doc 5 (RL) most, then doc 2 (DL).")
    print("But: 'AlphaGo' appears in doc 5, but embedding may prioritize")
    print("     general DL docs over the specific RL doc.")
    print("-" * 70)
    
    query_irrelevant = "AlphaGo使用的深度学习技术有哪些？"
    
    retrieved_i = rag.retrieve(query_irrelevant, top_k=3)
    print("\nRetrieved documents:")
    for i, doc in enumerate(retrieved_i):
        print(f"  [{i+1}] score={doc['score']:.4f}: {doc['content'][:80]}...")
    
    # Check if the RL doc (index 4 in 0-based, doc 5 in 1-based) is in top results
    rl_doc_text = test_documents[4][:50]
    rl_found = any(rl_doc_text in doc['content'] for doc in retrieved_i)
    print(f"\n  RL document (doc 5) retrieved? {'YES' if rl_found else 'NO - THIS IS THE FAILURE!'}")
    
    answer_i = rag.generate(query_irrelevant, retrieved_i)
    print(f"\nGENERATED ANSWER: {answer_i}")
    
    print("\nANALYSIS:")
    print("  - The embedding model sees '深度学习技术' and matches DL docs.")
    print("  - But the most relevant doc (RL, which mentions AlphaGo) may rank lower.")
    print("  - This happens because embedding models capture general semantics,")
    print("    not the specific factual relationship we need.")
    print("  - Solutions: reranking, HyDE, query expansion, better chunking.")
    
else:
    print("Skipping failure mode demonstrations - no API key available.")
    print("Set OPENAI_API_KEY and re-run the indexing cell, then this cell to see the demos.")

## 8. 总结与Phase 04预告

### 我们学到了什么

在本Notebook中，我们：

1. **从零构建了一个完整的Naive RAG系统**，包含索引、检索、生成三个核心阶段
2. **理解了每个阶段的输入、输出和关键设计决策**
3. **亲手写了余弦相似度计算、Prompt构建、API调用**——没有使用任何RAG框架
4. **观察了Naive RAG的三种典型失败模式**：幻觉、缺失上下文、不相关检索

### Naive RAG能做什么

- 在小规模文档集（< 1K文档）上进行基本的问答
- 当答案明确存在于单个文档中时，效果不错
- 作为原型验证RAG概念的可行性

### Naive RAG不能做什么

- 处理长文档（需要分块策略）
- 精确检索（需要重排序）
- 处理模糊查询（需要查询改写/HyDE）
- 大规模部署（需要向量数据库）
- 多跳推理（需要迭代检索）

### Phase 04 预告：RAG深度优化

在下一个Phase中，我们将学习如何将Naive RAG升级为**生产级RAG**：

| 优化维度 | Naive RAG | 进阶RAG | 生产级RAG |
|---------|----------|--------|----------|
| **分块** | 无 | 固定大小分块 | 语义分块 + 小-大窗口 |
| **检索** | 暴力余弦相似度 | 向量数据库 | 混合检索 + 重排序 |
| **查询** | 原样查询 | 查询改写 | HyDE + 多查询融合 |
| **生成** | 单次生成 | 系统提示词 | Self-RAG / CRAG |
| **评估** | 无 | 人工评估 | RAGAS自动化评估 |

准备好了吗？让我们进入Phase 04，把Naive RAG变成一个真正可用的系统！